# Version 2 — audit des étiquettes, observations temporelles et ablations

Ce notebook complète `pipeline_v2.ipynb`, à la suite d'un audit du protocole. Il traite quatre points :

1. **Étiquettes** : quel produit Copernicus a produit chaque étiquette, à quelle date, quelle partie de la zone a vraiment été observée,
   et quelle est la sensibilité des résultats au tampon de 0, 15 ou 30 m.
2. **Cohérence temporelle** : les variables de pluie sont reconstruites à plusieurs **instants de prédiction** *t*,
   avec uniquement l'information disponible à *t*.
3. **Ablations** par groupe de variables (terrain, pluie, bâti, drainage), avec les mêmes plis.
4. **Généralisation géographique** : un pays entier est exclu de l'entraînement.

### Objectif évalué (objectif A)

> **À l'instant *t*, avec la pluie observée jusqu'à *t* et une prévision des heures suivantes, quelles routes seront inondées pendant l'événement en cours ?**

Les étiquettes disent *si* une route a été inondée pendant l'événement, pas *quand*. Ce notebook **n'évalue donc pas** une alerte à horizon fixe
(« inondée dans les 20 prochaines minutes »), qui demanderait des étiquettes horodatées.

Scripts : `flood_risk/real/labels.py` (audit des étiquettes), `flood_risk/real/temporal.py` (observations temporelles),
`flood_risk/v2.py` (`FEATURE_SETS`), `flood_risk/train.py` (`cross_validate_times`).

In [1]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import json, time, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

from flood_risk.schema import LABEL, RAW_FEATURES, RELATIVE_FEATURES
from flood_risk.train import cross_validate, cross_validate_times, NAMES, MAIN_LEAD
from flood_risk.real.ems import EVENTS
from flood_risk.real.temporal import OFFSETS_H, FORECAST_SIGMA
from flood_risk import v2

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
OUT = Path("reports/med_temporal"); OUT.mkdir(parents=True, exist_ok=True)
PARAMS = json.loads(Path("models/med_imerg/hgb_params.json").read_text())
SEED = 0
COUNTRY = {"EMSR319": "Tunisie", "EMSR696": "Libye", "EMSR773": "Espagne", "EMSR388": "Espagne", "EMSR569": "Espagne",
           "EMSR327": "Espagne", "EMSR323": "Espagne", "EMSR324": "France", "EMSR257": "Grèce", "EMSR413": "Grèce",
           "EMSR480": "Grèce", "EMSR548": "Italie", "EMSR649": "Italie"}
CODE_HASH = hashlib.sha256(b"".join(Path(f"flood_risk/{m}.py").read_bytes() for m in ["v2", "features", "train"])).hexdigest()[:16]

def cached(name, settings, run):
    """Run an experiment once; reload its metrics and out-of-fold predictions while settings and model code are unchanged."""
    d = OUT / "cache" / name
    key = json.dumps({**settings, "code": CODE_HASH}, sort_keys=True, default=str)
    t0 = time.perf_counter()
    if (d / "key.json").exists() and (d / "key.json").read_text() == key:
        res, oof = pd.read_csv(d / "metrics.csv"), dict(np.load(d / "oof.npz"))
        print(f"{name} : relu depuis le cache ({time.perf_counter() - t0:.1f} s)")
    else:
        res, oof = run()
        d.mkdir(parents=True, exist_ok=True)
        res.to_csv(d / "metrics.csv", index=False); np.savez_compressed(d / "oof.npz", **oof); (d / "key.json").write_text(key)
        print(f"{name} : {time.perf_counter() - t0:.0f} s")
    return res, oof

## 1. Audit des étiquettes

### 1.1 Registre des produits Copernicus utilisés

Pour chaque zone : produit et version, capteurs et résolution, date des images après la crue, part de la zone réellement analysée
(couche *image footprint* moins les polygones *Not Analysed*), et part des routes inondées selon le tampon.

In [2]:
reg = pd.read_csv("reports/med_labels/product_registry.csv")
to_utc = lambda x: pd.to_datetime(x, utc=True, format="mixed").dt.tz_localize(None)   # some dates carry a time zone, some not
reg["image_first"] = to_utc(reg.image_first)
reg["pays"] = reg.event_id.map(COUNTRY)
peaks = pd.read_csv("data/inventory_med_temporal.csv.gz", usecols=["aoi", "offset_h", "prediction_time"])
peaks = peaks[peaks.offset_h == 0].groupby("aoi").prediction_time.first()
peaks = pd.Series(to_utc(peaks).values, index=peaks.index)
reg["jours pic de pluie → image"] = (reg.image_first - reg.aoi.map(peaks)).dt.total_seconds() / 86400
show = reg[["event_id", "pays", "aoi", "product", "sensors", "image_first", "jours pic de pluie → image", "segments",
            "observed_share", "flooded_b0", "flooded_b15", "flooded_b30"]]
print(f"date d'image connue pour {reg.image_first.notna().sum()} zones sur {len(reg)}")
show.round(3)

date d'image connue pour 11 zones sur 28


,event_id,pays,aoi,product,sensors,image_first,jours pic de pluie → image,segments,observed_share,flooded_b0,flooded_b15,flooded_b30
0,EMSR319,Tunisie,EMSR319_03KORBA,01DELINEATION_MAP_v1,SPOT-6-7 (1.5 m),2018-10-01 09:42:01,8.863,8649,1.000,0.002,0.004,0.005
1,EMSR319,Tunisie,EMSR319_05SOMAA,01DELINEATION_MAP_v2,SPOT-6-7 (1.5 m),NaT,NaN,7481,1.000,0.006,0.009,0.011
2,EMSR319,Tunisie,EMSR319_02MANZILBUZALAFAH,01DELINEATION_MAP_v2,SPOT-6-7 (1.5 m),2018-10-01 09:42:01,8.988,11981,1.000,0.001,0.006,0.009
3,EMSR319,Tunisie,EMSR319_01TAKLISAH,01DELINEATION_MAP_v2,SPOT-6-7 (1.5 m),2018-10-01 09:41:43,8.987,2607,0.947,0.001,0.002,0.003
4,EMSR696,Libye,EMSR696_AOI02,GRA v2,Pleiades (VHR1),2023-09-13 09:15:00,2.469,19558,1.000,0.111,0.184,0.221
5,EMSR696,Libye,EMSR696_AOI04,GRA v1,GeoEye-1 (VHR1),2023-09-13 09:18:00,2.096,6009,0.998,0.360,0.453,0.511
6,EMSR696,Libye,EMSR696_AOI07,GRA v1,WorldView-2 (VHR1),2023-09-13 09:53:00,2.537,839,0.995,0.908,0.915,0.921
7,EMSR773,Espagne,EMSR773_AOI03,DEL v1,GeoEye-1 (VHR2); WorldView-2 (VHR1),2024-10-31 10:22:00,1.807,20821,0.941,0.154,0.198,0.227
8,EMSR773,Espagne,EMSR773_AOI04,DEL v1,GeoEye-1 (VHR2),2024-10-31 10:22:00,2.224,8846,0.938,0.129,0.199,0.227
9,EMSR388,Espagne,EMSR388_AOI02,DEL_PRODUCT_r1_RTP01_v2,NaN,NaT,NaN,53770,0.694,0.026,0.055,0.082


In [3]:
lab = pd.read_csv("data/labels_med.csv.gz")
inv = pd.read_csv("data/inventory_med_imerg.csv.gz")
inv = inv.merge(lab.drop(columns="event_id"), on=["segment_id", "aoi"], how="left")
print("lignes appariées avec l'audit :", f"{inv.flooded_b15.notna().mean():.2%}")
print("même étiquette qu'à la construction (tampon 15 m) :", f"{(inv.flooded == inv.flooded_b15).mean():.3%}")
inv = inv.sort_values(LABEL).drop_duplicates(["segment_id", "event_id"], keep="last").sort_index()   # zones qui se chevauchent
summary = inv.groupby("event_id").agg(segments=(LABEL, "size"), observes=("observed", "mean"),
                                      b0=("flooded_b0", "mean"), b15=("flooded_b15", "mean"), b30=("flooded_b30", "mean"))
summary["b15 parmi observés"] = inv[inv.observed == 1].groupby("event_id").flooded_b15.mean()
summary.insert(0, "pays", summary.index.map(COUNTRY))
summary.round(4)

lignes appariées avec l'audit : 100.00%
même étiquette qu'à la construction (tampon 15 m) : 100.000%


,pays,segments,observes,b0,b15,b30,b15 parmi observés
event_id,,,,,,,
EMSR257,Grèce,2777,1.0000,0.1559,0.1732,0.1869,0.1732
EMSR319,Tunisie,30715,0.9955,0.0021,0.0060,0.0082,0.0060
EMSR323,Espagne,1990,0.9970,0.1206,0.2196,0.2543,0.2203
EMSR324,France,7120,1.0000,0.0160,0.0317,0.0468,0.0317
EMSR327,Espagne,1272,1.0000,0.0024,0.0228,0.0432,0.0228
EMSR388,Espagne,60540,0.7184,0.0660,0.1069,0.1337,0.1482
EMSR413,Grèce,4855,0.8350,0.1563,0.1775,0.1946,0.2092
EMSR480,Grèce,5732,0.9935,0.0569,0.1383,0.1577,0.1392
EMSR548,Italie,34351,0.7058,0.0013,0.0059,0.0093,0.0084


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
delay = reg.groupby("event_id")["jours pic de pluie → image"].min().reindex(summary.index)
axes[0].bar(summary.index, delay, color="#64748b")
axes[0].set(ylabel="jours entre le pic de pluie et l'image", title="Délai des images après la pluie (vide = date inconnue)")
axes[0].tick_params(axis="x", rotation=60, labelsize=8)
x = np.arange(len(summary))
for i, b in enumerate(["b0", "b15", "b30"]):
    axes[1].bar(x + (i - 1) * 0.27, 100 * summary[b], 0.27, label=f"tampon {b[1:]} m")
axes[1].set_xticks(x, summary.index, rotation=60, fontsize=8); axes[1].set_yscale("log")
axes[1].set(ylabel="% de routes inondées (log)", title="Sensibilité de l'étiquette au tampon"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Analyse de l'audit des étiquettes.**

- **Reconstruction exacte.** Les segments refaits à partir des fichiers en cache correspondent à 100 % aux lignes de l'inventaire, et l'étiquette à 15 m est identique à 100 %.
- **Dates d'image.** Elles ne sont connues que pour **11 zones sur 28** : les produits « RTP » (2019–2023) ne contiennent pas la date de l'image. Quand elle est connue, l'image a été prise après le pic de pluie :
  - **≈ 2 jours** pour Derna, Valence 2024 et Mallorca ;
  - **3,5 jours** pour Mandra ;
  - **6 jours** pour l'Andalousie ;
  - **≈ 9 jours pour Nabeul** (image SPOT du 1er octobre 2018, crue du 22 septembre).

  En crue éclair, l'eau se retire en quelques heures à quelques jours. Une partie des routes réellement inondées a donc pu être classée « sèche », surtout à Nabeul. C'est une explication plausible de son taux très bas (0,6 %).
- **Couverture.** La plupart des zones sont analysées à plus de 94 %. Trois crues ont une part importante de **zones non observées** (hors image ou « Not Analysed »), qui étaient jusqu'ici comptées comme sèches :
  - SE Espagne 2019 : 72 % observé ;
  - Sicile 2021 : 71 % ;
  - Attique ouest : 84 %.
- **Tampon.** Le taux d'inondés passe de 7,4 % (0 m) à 10,8 % (15 m) puis 12,8 % (30 m) sur l'ensemble, mais il **triple à Nabeul** (0,21 % → 0,60 % → 0,82 %).
  Beaucoup de routes de Nabeul sont étiquetées inondées parce qu'elles **longent** une zone d'eau, et non parce qu'elles la traversent.

### 1.2 Effet des choix d'étiquetage sur les résultats

Même modèle (V2 gradient boosting, paramètres réglés), même protocole que `pipeline_v2.ipynb` (13 crues, une crue testée à la fois,
Élevé = 2 % de chaque zone, délai 20 min). Seule l'étiquette change :

- **15 m** : la version actuelle ;
- **15 m, zones observées seulement** : les routes hors de l'image analysée sont retirées (entraînement et test) ;
- **0 m** et **30 m** : tampon plus strict ou plus large.

In [5]:
COLS = list(dict.fromkeys(RAW_FEATURES + RELATIVE_FEATURES + [LABEL, "event_id", "segment_id", "aoi", "rain_interpolated",
                                                                    "observed", "flooded_b0", "flooded_b15", "flooded_b30"]))
base_df = inv[[c for c in COLS if c in inv.columns]].copy()
f64 = base_df.select_dtypes("float64").columns; base_df[f64] = base_df[f64].astype("float32")
variants = {"tampon 15 m": ("flooded_b15", False), "15 m, zones observées": ("flooded_b15", True),
            "tampon 0 m": ("flooded_b0", False), "tampon 30 m": ("flooded_b30", False)}
label_rows = []
for name, (col, observed_only) in variants.items():
    d = base_df[base_df.observed == 1] if observed_only else base_df
    d = d.assign(flooded=d[col].astype(int)).reset_index(drop=True)
    res_l, _ = cached(f"labels_{col}_{int(observed_only)}", {"variant": name, "rows": len(d), "params": PARAMS},
                      lambda d=d: cross_validate(d, d.event_id.nunique(), SEED, 0.3, PARAMS, "area", ["hgb", "hgb_terrain"]))
    m = res_l[(res_l.target == LABEL) & (res_l.lead == MAIN_LEAD)]
    for k in ["hgb", "hgb_terrain"]:
        mk = m[m.model == k]
        label_rows.append({"étiquette": name, "modèle": NAMES[k], "lignes": len(d), "taux d'inondés": d.flooded.mean(),
                           **mk[["roc_auc", "pr_auc", "recall_high", "false_alarm", "bss"]].mean().to_dict(),
                           "PR-AUC / taux": (mk.pr_auc / mk.events.map(d.groupby("event_id").flooded.mean())).mean(),
                           "Nabeul ROC": mk[mk.events == "EMSR319"].roc_auc.mean()})
pd.DataFrame(label_rows).set_index(["étiquette", "modèle"]).round(3)

fold 1/13 done (1 test events)


fold 2/13 done (1 test events)


fold 3/13 done (1 test events)


fold 4/13 done (1 test events)


fold 5/13 done (1 test events)


fold 6/13 done (1 test events)


... (43 outputs left out)

fold 11/13 done (1 test events)


fold 12/13 done (1 test events)


fold 13/13 done (1 test events)
labels_flooded_b30_0 : 337 s


lignes  taux d'inondés  roc_auc  \
étiquette             modèle                                                  
tampon 15 m           V2 gradient boosting  216171           0.108    0.737   
                      V2 GB terrain only    216171           0.108    0.739   
15 m, zones observées V2 gradient boosting  186243           0.125    0.724   
                      V2 GB terrain only    186243           0.125    0.729   
tampon 0 m            V2 gradient boosting  216171           0.074    0.730   
                      V2 GB terrain only    216171           0.074    0.736   
tampon 30 m           V2 gradient boosting  216171           0.128    0.731   
                      V2 GB terrain only    216171           0.128    0.736   

                                            pr_auc  recall_high  false_alarm  \
étiquette             modèle                                                   
tampon 15 m           V2 gradient boosting   0.206        0.100        0.734   
                      V2 GB terrain only     0.242        0.110        0.714   
15 m, zones observées V2 gradient boosting   0.211        0.099        0.715   
                      V2 GB terrain only     0.249        0.108        0.689   
tampon 0 m            V2 gradient boosting   0.139        0.103        0.808   
                      V2 GB terrain only     0.166        0.080        0.808   
tampon 30 m           V2 gradient boosting   0.241        0.099        0.688   
                      V2 GB terrain only     0.282        0.104        0.669   

                                              bss  PR-AUC / taux  Nabeul ROC  
étiquette             modèle                                                  
tampon 15 m           V2 gradient boosting  0.015          3.518       0.809  
                      V2 GB terrain only   -0.021          4.625       0.789  
15 m, zones observées V2 gradient boosting  0.021          3.332       0.822  
                      V2 GB terrain only   -0.019          4.213       0.790  
tampon 0 m            V2 gradient boosting  0.010          4.033       0.838  
                      V2 GB terrain only   -0.036          4.934       0.802  
tampon 30 m           V2 gradient boosting  0.012          3.390       0.809  
                      V2 GB terrain only   -0.023          4.322       0.796

**Analyse — les résultats résistent-ils aux choix d'étiquetage ?** Oui pour le classement, non pour les valeurs absolues.

- **Le ROC-AUC bouge peu** : il reste entre 0,72 et 0,74 pour V2 et entre 0,73 et 0,74 pour le terrain seul, quel que soit le tampon. Le **classement des routes** ne dépend donc pas du choix de 15 m.
- **Le PR-AUC suit le taux d'inondés** : 0,14 à 0 m, 0,21 à 15 m, 0,24 à 30 m pour V2. Rapporté au taux de base, il reste stable (3,3 à 4,0 pour V2, 4,2 à 4,9 pour le terrain).
  Les PR-AUC de tampons différents ne doivent donc **pas** être comparés directement.
- **Retirer les zones non observées aide un peu** : moins de fausses alertes (73 % → 72 % pour V2, 71 % → 69 % pour le terrain), et Nabeul passe de 0,81 à 0,82.
  C'est l'étiquetage le plus honnête. **Il est recommandé pour les prochains entraînements.**
- **Les conclusions entre modèles ne changent pas** :
  - le terrain seul garde un meilleur PR-AUC que V2 dans les quatre variantes ;
  - V2 garde un meilleur ROC-AUC à Nabeul (0,81 à 0,84 contre 0,79 à 0,80).

## 2. Observations temporelles

`flood_risk/real/temporal.py` crée, pour chaque route, une ligne par **instant de prédiction** *t* : 12 h, 6 h, 3 h et 1 h avant
l'heure la plus pluvieuse de sa zone, et à cette heure même (offsets [-12, -6, -3, -1, 0]).

| Variable | Information utilisée | Disponible à *t* ? |
|---|---|---|
| `rain_1h_mm`, `rain_3h_mm`, `rain_24h_mm` | pluie **observée** dans l'heure, les 3 h, les 24 h qui se terminent à *t* | oui (en opération : pluviomètres, radar ou IMERG Early) |
| `rain_event_mm` | pluie observée depuis le début de l'orage jusqu'à *t* | oui |
| `dry_spell_days` | jours secs avant l'orage | oui |
| `rain_eta_mmh`, `rain_next_6h_mm` | pluie de l'heure suivante et des 6 h suivantes, **prévision simulée** (vraie pluie × erreur log-normale, σ = 0.5 et 0.7) | oui si une prévision existe ; ici simulée, faute de prévisions archivées |

L'heure la plus pluvieuse de la zone est trouvée après coup. Elle sert **seulement à placer les instants** et n'entre dans aucune variable.
L'étiquette reste « route inondée pendant l'événement ».

In [6]:
tmp = pd.read_csv("data/inventory_med_temporal.csv.gz")
tmp = tmp.sort_values(LABEL).drop_duplicates(["segment_id", "event_id", "offset_h"], keep="last").sort_index().reset_index(drop=True)
f64 = tmp.select_dtypes("float64").columns; tmp[f64] = tmp[f64].astype("float32")
print(f"{len(tmp):,} lignes = {tmp.groupby('event_id').segment_id.nunique().sum():,} routes x {tmp.offset_h.nunique()} instants")
rain_cols = ["rain_1h_mm", "rain_3h_mm", "rain_24h_mm", "rain_event_mm", "rain_eta_mmh", "rain_next_6h_mm"]
print(tmp.groupby("offset_h")[rain_cols].mean().round(2).to_string())
# Contrôle d'absence de fuite : les cumuls observés ne peuvent pas décroître quand t avance
per_seg = tmp.groupby(["event_id", "segment_id", "offset_h"]).rain_event_mm.first().unstack()
print("rain_event_mm croissant avec t pour", f"{(per_seg.diff(axis=1).iloc[:, 1:] >= -1e-3).all(axis=1).mean():.2%}", "des routes")

1,080,855 lignes = 216,171 routes x 5 instants
          rain_1h_mm  rain_3h_mm  rain_24h_mm  rain_event_mm  rain_eta_mmh  rain_next_6h_mm
offset_h                                                                                   
-12             1.26        5.47    15.540000      18.740000          1.42        11.850000
-6              1.91        4.31    26.540001      30.080000          3.72        31.000000
-3              2.71        7.59    33.900002      37.480000          5.12        38.500000
-1              7.75       15.64    46.320000      49.570000         11.97        30.629999
 0             11.79       24.77    57.250000      60.189999          7.56        28.299999


rain_event_mm croissant avec t pour 99.95% des routes


In [7]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
g = tmp.groupby(["offset_h", LABEL])[["rain_event_mm", "rain_next_6h_mm"]].mean().unstack()
for lab, c in [(0, "#94a3b8"), (1, "#dc2626")]:
    ax[0].plot(g.index, g[("rain_event_mm", lab)], marker="o", color=c, label="inondée" if lab else "sèche")
    ax[1].plot(g.index, g[("rain_next_6h_mm", lab)], marker="o", color=c, label="inondée" if lab else "sèche")
ax[0].set(xlabel="heures avant la pluie la plus forte de la zone", ylabel="mm", title="Pluie observée depuis le début de l'orage")
ax[1].set(xlabel="heures avant la pluie la plus forte de la zone", ylabel="mm", title="Pluie prévue pour les 6 h suivantes")
ax[0].legend(); plt.tight_layout(); plt.show()

**Analyse des observations temporelles.**

- **Taille.** 1 080 855 lignes, soit 216 171 routes × 5 instants.
- **La pluie évolue comme attendu.** La pluie observée depuis le début de l'orage passe en moyenne de 19 mm (12 h avant le pic) à 60 mm (au pic). La prévision des 6 h suivantes culmine 3 h avant le pic (38 mm).
- **Contrôle de non-fuite.** Le cumul observé ne décroît jamais quand *t* avance, pour 99,95 % des routes. Les 0,05 % restants viennent de la fusion des zones qui se chevauchent : une route peut prendre la pluie de l'autre zone.
- **Pluie des routes inondées et des routes sèches.** Sur l'ensemble, les routes inondées ont reçu environ 50 % de pluie en plus (87 contre 58 mm au pic, graphique). Mais **à l'intérieur d'une même zone**, l'écart n'est plus que de **+2 %** en médiane (18 zones sur 28 vont dans ce sens). La pluie distingue donc les **zones** entre elles, et presque pas les routes d'une même zone.

## 3. Ablations par groupe de variables (une crue testée à la fois)

| Expérience | Variables |
|---|---|
| **A** | terrain : altitude relative, pente, cuvettes, TWI, distance à l'oued, percentiles, passages, gués |
| **B** | pluie seule : observée jusqu'à *t* + prévue après *t* |
| **C** | terrain + pluie |
| **D** | terrain + pluie + bâti (WorldCover) |
| **D sans prévision** | terrain + bâti + pluie **observée seulement** |
| **E** | toutes les variables, dont les indicateurs de drainage |
| **E sans drainage** | toutes les variables sauf les indicateurs de drainage |

Tous les modèles : gradient boosting monotone avec les mêmes paramètres, calibration et niveaux identiques (Élevé = 2 % de chaque zone
**à chaque instant**), mêmes plis. Les colonnes constantes sont retirées automatiquement.

In [8]:
SETS = list(v2.FEATURE_SETS)
res_t, oof_t = cached("ablations_event", {"sets": SETS, "rows": len(tmp), "params": PARAMS, "neg": 0.15},
                      lambda: cross_validate_times(tmp, SETS, "event_id", SEED, 0.15, PARAMS))
res_t["événement"] = res_t.group
cols = ["roc_auc", "pr_auc", "recall_high", "false_alarm", "brier", "bss"]
tab = res_t.groupby(["model", "offset_h"])[cols].mean()
print("Moyenne des 13 crues, par instant de prédiction (h avant la pluie la plus forte) :")
tab.loc[SETS].round(3)

fold 1: EMSR388 done


fold 2: EMSR548 done


fold 3: EMSR319 done


fold 4: EMSR773 done


fold 5: EMSR696 done


fold 6: EMSR569 done


... (5 outputs left out)

fold 12: EMSR327 done


fold 13: EMSR649 done


ablations_event : 1979 s
Moyenne des 13 crues, par instant de prédiction (h avant la pluie la plus forte) :


roc_auc  pr_auc  recall_high  false_alarm  \
model                     offset_h                                              
A_terrain                 -12         0.749   0.258        0.121        0.704   
                          -6          0.749   0.258        0.121        0.704   
                          -3          0.749   0.259        0.122        0.699   
                          -1          0.749   0.259        0.119        0.704   
                           0          0.750   0.260        0.121        0.700   
B_rain                    -12         0.492   0.131        0.012        0.913   
                          -6          0.494   0.132        0.014        0.901   
                          -3          0.596   0.176        0.058        0.771   
                          -1          0.486   0.144        0.019        0.894   
                           0          0.511   0.140        0.013        0.911   
C_terrain_rain            -12         0.741   0.243        0.107        0.727   
                          -6          0.738   0.262        0.118        0.699   
                          -3          0.765   0.269        0.119        0.696   
                          -1          0.739   0.236        0.105        0.701   
                           0          0.725   0.225        0.111        0.704   
D_terrain_rain_impervious -12         0.743   0.237        0.108        0.731   
                          -6          0.735   0.254        0.121        0.696   
                          -3          0.754   0.258        0.120        0.693   
                          -1          0.737   0.228        0.109        0.704   
                           0          0.729   0.226        0.130        0.690   
D_observed_only           -12         0.741   0.234        0.107        0.727   
                          -6          0.724   0.237        0.130        0.700   
                          -3          0.739   0.239        0.130        0.689   
                          -1          0.727   0.223        0.129        0.693   
                           0          0.733   0.229        0.132        0.692   
E_full                    -12         0.740   0.234        0.108        0.729   
                          -6          0.735   0.253        0.121        0.704   
                          -3          0.753   0.256        0.116        0.709   
                          -1          0.736   0.224        0.107        0.711   
                           0          0.727   0.223        0.123        0.698   
E_no_drainage             -12         0.739   0.230        0.102        0.742   
                          -6          0.740   0.255        0.116        0.711   
                          -3          0.754   0.261        0.114        0.702   
                          -1          0.737   0.226        0.102        0.711   
                           0          0.730   0.223        0.126        0.698   

                                    brier    bss  
model                     offset_h                
A_terrain                 -12       0.102  0.004  
                          -6        0.102  0.004  
                          -3        0.102  0.004  
                          -1        0.102  0.006  
                           0        0.102  0.006  
B_rain                    -12       0.102 -0.031  
                          -6        0.103 -0.097  
                          -3        0.103 -0.096  
                          -1        0.104 -0.104  
                           0        0.103 -0.097  
C_terrain_rain            -12       0.102  0.018  
                          -6        0.102 -0.006  
                          -3        0.102 -0.014  
                          -1        0.104 -0.022  
                           0        0.103 -0.006  
D_terrain_rain_impervious -12       0.102  0.008  
                          -6        0.102 -0.009  
                          -3        0.102 -0.018  
                  

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors = dict(zip(SETS, ["#16a34a", "#7c3aed", "#2563eb", "#dc2626", "#f59e0b", "#0f172a", "#94a3b8"]))
for k in SETS:
    for ax, c in zip(axes, ["roc_auc", "pr_auc", "recall_high"]):
        ax.plot(tab.loc[k].index, tab.loc[k, c], marker="o", color=colors[k], label=NAMES[k])
for ax, c in zip(axes, ["ROC-AUC", "PR-AUC", "rappel Élevé (2 %)"]):
    ax.set(xlabel="heures avant la pluie la plus forte", ylabel=c, title=c)
axes[0].legend(fontsize=7); plt.tight_layout(); plt.show()

**Analyse des ablations** (moyenne des 13 crues, une crue testée à la fois, seulement l'information disponible à *t*).

| Expérience | ROC-AUC (−12 h → pic) | PR-AUC à −3 h | Rappel Élevé à −3 h |
|---|---|---|---|
| **A terrain** | **0,75 stable** | **0,26** | 12,2 % |
| B pluie seule | 0,49 → 0,51 (0,60 à −3 h) | 0,18 | 5,8 % |
| C terrain + pluie | 0,74 → 0,73 (0,77 à −3 h) | 0,27 | 11,9 % |
| D + bâti | 0,74 → 0,73 | 0,26 | 12,0 % |
| D sans prévision | 0,74 → 0,73 | 0,24 | 13,0 % |
| E complet | 0,74 → 0,73 | 0,26 | 11,6 % |
| E sans drainage | 0,74 → 0,73 | 0,26 | 11,4 % |

1. **Avec des variables honnêtes, la pluie n'améliore plus le classement de façon fiable.**
   - Le terrain seul (A) fait aussi bien que tous les modèles avec pluie, à tous les instants.
   - Les tests appariés (tableau suivant) ne montrent **aucune différence significative** : tous les p de Wilcoxon sont supérieurs à 0,15, et les gains se répartissent à peu près moitié-moitié entre crues.
2. **Le classement est bon dès 12 h avant le pic** (ROC-AUC 0,75). C'est logique, puisqu'il repose surtout sur le terrain, connu à l'avance.
3. **La prévision de pluie n'apporte pas de gain mesurable** (D contre D sans prévision : écarts autour de 0, non significatifs).
4. **Les indicateurs de drainage n'apportent rien** (E contre E sans drainage : non significatif).
5. **Probabilités** : le BSS reste proche de 0 pour tous les modèles (entre −0,10 et +0,04). Comme dans `pipeline_v2.ipynb`, les modèles classent mais ne donnent pas de probabilité fiable.

**Ce qui change par rapport à `pipeline_v2.ipynb`** :
- Le léger avantage de V2 complet sur le terrain seul (Nabeul, crues étendues) venait de variables de pluie calculées **à l'heure du pic, choisie après coup**.
- Quand la pluie ne contient que ce qui était connu à *t*, cet avantage **n'est plus démontré**.
- C'est exactement le risque signalé par l'audit.

In [10]:
def paired(a, b, metric, off):
    x = res_t[(res_t.offset_h == off)].pivot_table(index="group", columns="model", values=metric)
    d = x[a] - x[b]
    return {"comparaison": f"{NAMES[a]} − {NAMES[b]}", "instant": off, "métrique": metric, "écart moyen": d.mean(),
            "médiane": d.median(), "crues gagnées": f"{(d > 0).sum()}/{len(d)}",
            "Wilcoxon p": wilcoxon(d).pvalue if (d != 0).any() else 1.0, "Nabeul": d.get("EMSR319", np.nan)}
pairs = [("C_terrain_rain", "A_terrain"), ("D_terrain_rain_impervious", "C_terrain_rain"),
         ("D_terrain_rain_impervious", "D_observed_only"), ("E_full", "E_no_drainage"), ("E_full", "D_terrain_rain_impervious")]
pd.DataFrame([paired(a, b, m, off) for a, b in pairs for off in (-6, 0) for m in ("pr_auc", "recall_high")]).round(4)

,comparaison,instant,métrique,écart moyen,médiane,crues gagnées,Wilcoxon p,Nabeul
0,C terrain + rain − A terrain,-6,pr_auc,0.0039,0.0027,7/13,0.8926,-0.0953
1,C terrain + rain − A terrain,-6,recall_high,-0.0031,0.0000,6/13,0.8457,0.0272
2,C terrain + rain − A terrain,0,pr_auc,-0.0349,-0.0097,5/13,0.3054,-0.0763
3,C terrain + rain − A terrain,0,recall_high,-0.0100,-0.0021,4/13,0.2036,-0.0054
4,D terrain + rain + built-up − C terrain + rain,-6,pr_auc,-0.0086,-0.0096,4/13,0.3396,-0.0106
5,D terrain + rain + built-up − C terrain + rain,-6,recall_high,0.0033,0.0000,4/13,0.4922,-0.0652
6,D terrain + rain + built-up − C terrain + rain,0,pr_auc,0.0007,-0.0070,5/13,0.7354,-0.0204
7,D terrain + rain + built-up − C terrain + rain,0,recall_high,0.0184,0.0000,6/13,0.6377,-0.0109
8,D terrain + rain + built-up − D without foreca...,-6,pr_auc,0.0167,-0.0004,6/13,0.9460,-0.0052
9,D terrain + rain + built-up − D without foreca...,-6,recall_high,-0.0091,-0.0003,5/13,0.4238,-0.0272


In [11]:
per_ev = res_t[res_t.offset_h == -3].pivot_table(index="group", columns="model", values="roc_auc")[SETS]
per_ev.index = [f"{e} {COUNTRY[e]}" for e in per_ev.index]
fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(per_ev.values, cmap="RdYlGn", vmin=0.3, vmax=1, aspect="auto")
ax.set_xticks(range(len(SETS)), [NAMES[k] for k in SETS], rotation=25, ha="right", fontsize=8)
ax.set_yticks(range(len(per_ev)), per_ev.index, fontsize=8)
for i in range(per_ev.shape[0]):
    for j in range(per_ev.shape[1]):
        ax.text(j, i, f"{per_ev.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.set_title("ROC-AUC par crue, 3 h avant la pluie la plus forte"); plt.colorbar(im); plt.tight_layout(); plt.show()

**Analyse crue par crue (3 h avant le pic).**

- **La pluie aide nettement sur deux crues** :
  - Sicile 2023 : 0,72 → 0,86 ;
  - Mandra : 0,36 → 0,43, même si le score reste mauvais.
- **Elle aide un peu** sur Valence 2024 (0,71 → 0,75) et SE Espagne (0,75 → 0,78, seulement en C).
- **Elle nuit** sur Mallorca (0,66 → 0,60) et Valence 2022 (0,86 → 0,81).
- **Nabeul** :
  - le ROC-AUC monte légèrement avec la pluie (0,78 → 0,79 à 0,82 selon le modèle et l'instant) ;
  - mais le rappel au niveau Élevé est **meilleur avec le terrain seul** (32,6 % contre 27 à 30 % pour D et E), sauf pour C à −6 h (35 %).
- **Derna** reste au hasard (0,49–0,50) pour tous les modèles.

Les effets de la pluie sont donc **réels mais inconstants**, et dans les deux sens : 13 crues ne suffisent pas pour qu'un modèle apprenne quand la pluie compte.

## 4. Généralisation géographique : un pays entier exclu

Plusieurs crues touchent la même région (Valence 2022 et 2024, Mandra et Attique ouest, deux crues en Sicile), donc exclure une crue ne suffit pas
à tester une région inconnue. Ici, chaque pays est exclu en entier (6 plis : Espagne 5 crues, Grèce 3, Italie 2, France, Libye, Tunisie 1 chacune).

In [12]:
print(tmp.assign(pays=tmp.event_id.map(COUNTRY)).groupby("pays").agg(crues=("event_id", "nunique"), lignes=(LABEL, "size"), taux=(LABEL, "mean")).round(4))
tmp_c = tmp.assign(country=tmp.event_id.map(COUNTRY))
LOCO_SETS = ["A_terrain", "D_terrain_rain_impervious", "E_full"]
res_c, _ = cached("loco", {"sets": LOCO_SETS, "rows": len(tmp_c), "params": PARAMS, "neg": 0.15},
                  lambda: cross_validate_times(tmp_c, LOCO_SETS, "country", SEED, 0.15, PARAMS))
loco = res_c[res_c.offset_h == -3].pivot_table(index="group", columns="model", values=["roc_auc", "pr_auc", "recall_high"])
ev3 = res_t[res_t.offset_h == -3]
comp = pd.DataFrame({k: {"une crue exclue": ev3[ev3.model == k].roc_auc.mean(),
                         "un pays exclu": res_c[(res_c.offset_h == -3) & (res_c.model == k)].roc_auc.mean()} for k in LOCO_SETS}).T
comp.index = [NAMES[k] for k in LOCO_SETS]
display(loco.round(3))
comp.round(3)

         crues  lignes    taux
pays                          
Espagne      5  514800  0.1308
France       1   35600  0.0317
Grèce        3   66820  0.1598
Italie       2  178030  0.0070
Libye        1  132030  0.2682
Tunisie      1  153575  0.0060


fold 1: Espagne done


fold 2: Italie done


fold 3: Tunisie done


fold 4: Libye done


fold 5: Grèce done


fold 6: France done


loco : 391 s


pr_auc                                  recall_high  \
model   A_terrain D_terrain_rain_impervious E_full   A_terrain   
group                                                            
Espagne     0.309                     0.195  0.198       0.079   
France      0.340                     0.332  0.348       0.270   
Grèce       0.411                     0.425  0.422       0.060   
Italie      0.106                     0.106  0.107       0.333   
Libye       0.261                     0.251  0.250       0.026   
Tunisie     0.130                     0.025  0.027       0.326   

                                           roc_auc                            \
model   D_terrain_rain_impervious E_full A_terrain D_terrain_rain_impervious   
group                                                                          
Espagne                     0.058  0.058     0.726                     0.564   
France                      0.261  0.261     0.937                     0.932   
Grèce                       0.053  0.053     0.731                     0.687   
Italie                      0.321  0.317     0.926                     0.939   
Libye                       0.024  0.024     0.499                     0.491   
Tunisie                     0.288  0.293     0.784                     0.793   

                
model   E_full  
group           
Espagne  0.575  
France   0.934  
Grèce    0.688  
Italie   0.938  
Libye    0.488  
Tunisie  0.810

,une crue exclue,un pays exclu
A terrain,0.749,0.767
D terrain + rain + built-up,0.754,0.734
E full (with drainage proxies),0.753,0.739


**Analyse — un pays entier exclu.**

| Modèle | Une crue exclue | Un pays exclu |
|---|---|---|
| **A terrain** | 0,749 | **0,767** |
| D terrain + pluie + bâti | 0,754 | 0,734 |
| E complet | 0,753 | 0,739 |

- **Le terrain se transfère d'un pays à l'autre** : son score ne baisse pas quand un pays entier est exclu. Il monte même légèrement, mais ce n'est pas une différence significative.
- **Les modèles avec pluie perdent 0,015 à 0,02.**
  - La perte vient surtout de l'**Espagne** : quand ses 5 crues sont exclues, D tombe à 0,56 et E à 0,58, contre 0,73 pour le terrain seul.
  - La relation pluie → inondation apprise sur l'Espagne, la Grèce ou la Libye ne vaut pas ailleurs, car les pluies IMERG et les taux d'inondés y sont très différents.
- **Tunisie exclue.** Le modèle est appris sur les 12 crues des 5 autres pays :
  - ROC-AUC : 0,78 pour le terrain, 0,79 à 0,81 avec la pluie ;
  - rappel au niveau Élevé : 33 % pour le terrain, 29 % avec la pluie ;
  - PR-AUC : 0,13 pour le terrain, 0,03 avec la pluie.

  Le **haut du classement** est donc nettement meilleur avec le terrain seul.
- **Limite** : avec seulement 1 à 5 crues par pays, ces chiffres restent indicatifs.

## 5. Conclusion

### Ce que montre cet audit

1. **Étiquettes.**
   - Les cartes Copernicus sont prises **2 à 9 jours après la pluie** quand la date est connue, ce qui fait manquer des inondations courtes, surtout à Nabeul.
   - Jusqu'à 30 % de certaines zones **n'ont pas été observées**.
   - Le **classement** des routes résiste au choix du tampon (0, 15 ou 30 m) et au retrait des zones non observées. Les valeurs absolues de PR-AUC, elles, en dépendent.
2. **Cohérence temporelle.**
   - Une fois la pluie reconstruite avec seulement l'information disponible à *t*, **le terrain seul fait aussi bien que tous les modèles avec pluie**.
   - Aucune différence significative sur 13 crues, et un classement déjà bon 12 h avant le pic.
   - L'avantage de V2 complet vu dans `pipeline_v2.ipynb` dépendait en partie d'une pluie prise à l'heure du pic choisie après coup.
3. **Généralisation géographique.**
   - Le terrain se transfère à un pays jamais vu (ROC-AUC 0,77).
   - Les modèles avec pluie se transfèrent moins bien, surtout vers l'Espagne.
   - Pour la Tunisie exclue, le terrain seul garde le meilleur haut de classement.
4. **Drainage et bâti** : pas d'apport démontré.

### Conséquences pour le modèle

- **Pour classer les routes à l'intérieur d'une zone**, le modèle le plus solide aujourd'hui est le **terrain seul (A)** : il est aussi bon, plus simple, se transfère mieux et ne dépend d'aucune donnée météo en temps réel.
- **La pluie reste nécessaire**, mais à une autre échelle. Elle doit **déclencher l'alerte d'une zone**, puisqu'il n'y a pas de risque sans pluie, au lieu de servir à classer les routes d'une même zone.
  - Ce déclencheur ne peut pas être appris avec cet inventaire, qui ne contient que des orages ayant causé des crues.
  - Il doit être fixé par des seuils de pluie (INM, retours d'expérience), puis validé sur des orages sans inondation.
- **Aucune des expériences ne valide une alerte à +20 min.** Il faudrait pour cela des étiquettes horodatées : signalements datés de l'application (FR-21), caméras, ou relevés de la protection civile.

### Limites

- 13 crues, dont une seule tunisienne.
- Prévision de pluie simulée, faute de prévisions archivées.
- IMERG à 10 km.
- Dates d'image inconnues pour 17 zones sur 28.
- Les comparaisons entre modèles ne sont pas significatives, ce qui veut aussi dire qu'on ne peut pas exclure de petits gains de la pluie.